In [6]:
# автоматическая подгрузка измененных модулей
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from events_data_core.location_recognition import extract_loc

# 103-й старт (STS-102) по программе Спейс Шаттл. 29-й полёт шаттла Дискавери. Экипаж — Джеймс Уэзерби, Джеймс Келли, Эндрю Томас, Пол Ричардс, Джеймс Восс, Сьюзан Хэлмс, Юрий Усачёв(Россия).


text = (
    "103-й старт (STS-102) по программе Спейс Шаттл. 29-й полёт шаттла Дискавери. Экипаж — Джеймс Уэзерби, Джеймс Келли, Эндрю Томас, Пол Ричардс, Джеймс Восс, Сьюзан Хэлмс, Юрий Усачёв(Россия)")
result = extract_loc(text)

result

['Дискавери', 'Россия']

In [13]:
import pandas as pd

df = pd.read_csv("../data/events/2_struct/2000-2025.csv")

# === 4. Добавляем новую колонку ===
df["loc"] = df["event"].apply(extract_loc)

# === 5. Проверяем результат ===
df[["event", "loc"]].head()

,event,loc
0,Деноминация белорусского рубля;,[]
1,"В связи с «проблемой-2000», в Иране объявлен н...",[Иране]
2,"Вступление в силу закона в Великобритании, сог...",[Великобритании]
3,крушение украинского сухогруза типа «река-море...,"[Тикси, Николаев, Камбоджи, Туапсе]"
4,Обстрел из гранатомёта территории российского ...,"[Ливане, Бейрут]"


In [9]:
from events_data_core.country_normalization import normalize_country

words = [
    "Иране",
    "Великобритании",
    "Украине",
    "Франции",
    "Чечне",
    "Москве",
]

for w in words:
    print(f"{w:20} → {normalize_country(w)}")

Иране                → Иран
Великобритании       → Великобритания
Украине              → Украина
Франции              → Франция
Чечне                → Чечня
Москве               → Москва


In [10]:
from events_data_core.country_normalization import normalize_list
import pandas as pd

df = pd.read_csv("../data/events/3_countries/2000-2025.csv")
df['norm_loc'] = df["loc"].apply(normalize_list)
df.to_csv('../data/events/3_countries/2000-2025.csv', index=False, encoding="utf-8")
# todo здесь преобразование прилагательных в существительные Новосибирский -> Новосибирск, Тукрменский -> Туркмения. Возможно стоить поручить эту задачу ллм.
df

,date_start,date_end,event,loc,norm_loc
0,2000-01-01,NaN,Деноминация белорусского рубля;,[],[]
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н...",['Иране'],['Иран']
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог...",['Великобритании'],['Великобритания']
3,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...,"['Тикси', 'Николаев', 'Камбоджи', 'Туапсе']","['Тикси', 'Николаев', 'Камбоджа', 'Туапсе']"
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...,"['Ливане', 'Бейрут']","['Ливана', 'Бейрут']"
...,...,...,...,...,...
5625,2025-09-18,NaN,на Камчатке зафиксировано землетрясение магнит...,['Камчатке'],['Камчатка']
5626,2025-09-20,NaN,проведение конкурса песни «Интервидение» в Мос...,['Москве'],['Москва']
5627,2025-09-23,NaN,Международный уголовный суд представил подтвер...,['Филиппин'],['Филиппины']
5628,2025-09-25,NaN,Парламент Кыргызстана объявил о самороспуске.,['Кыргызстана'],['Кыргызстан']


In [11]:
COUNTRIES = [
    "Австралия", "Австрия", "Азербайджан", "Албания", "Алжир", "Ангола", "Андорра", "Антигуа и Барбуда",
    "Аргентина", "Армения", "Афганистан", "Багамы", "Бангладеш", "Барбадос", "Бахрейн", "Беларусь",
    "Белиз", "Бельгия", "Бенин", "Болгария", "Боливия", "Босния и Герцеговина", "Ботсвана", "Бразилия",
    "Бруней", "Буркина-Фасо", "Бурунди", "Бутан", "Вануату", "Ватикан", "Великобритания", "Венгрия",
    "Венесуэла", "Восточный Тимор", "Вьетнам", "Габон", "Гаити", "Гайана", "Гамбия", "Гана",
    "Гватемала", "Гвинея", "Гвинея-Бисау", "Германия", "Гондурас", "Гренада", "Греция", "Грузия",
    "Дания", "Джибути", "Доминика", "Доминиканская Республика", "Египет", "Замбия", "Зимбабве",
    "Израиль", "Индия", "Индонезия", "Иордания", "Ирак", "Иран", "Ирландия", "Исландия", "Испания",
    "Италия", "Йемен", "Кабо-Верде", "Казахстан", "Камбоджа", "Камерун", "Канада", "Катар", "Кения",
    "Кипр", "Киргизия", "Кирибати", "Китай", "Колумбия", "Коморы", "Конго", "ДР Конго", "Корея, Северная",
    "Корея, Южная", "Коста-Рика", "Кот-д’Ивуар", "Куба", "Кувейт", "Лаос", "Латвия", "Лесото",
    "Либерия", "Ливан", "Ливия", "Литва", "Лихтенштейн", "Люксембург", "Маврикий", "Мавритания",
    "Мадагаскар", "Малави", "Малайзия", "Мали", "Мальдивы", "Мальта", "Марокко", "Маршалловы Острова",
    "Мексика", "Микронезия", "Мозамбик", "Молдова", "Монако", "Монголия", "Мьянма", "Намибия", "Науру",
    "Непал", "Нигер", "Нигерия", "Нидерланды", "Никарагуа", "Новая Зеландия", "Норвегия",
    "Объединённые Арабские Эмираты",
    "Оман", "Пакистан", "Палау", "Панама", "Папуа — Новая Гвинея", "Парагвай", "Перу", "Польша",
    "Португалия", "Россия", "Руанда", "Румыния", "Сальвадор", "Самоа", "Сан-Марино", "Сан-Томе и Принсипи",
    "Саудовская Аравия", "Северная Македония", "Сейшелы", "Сенегал", "Сент-Винсент и Гренадины",
    "Сент-Китс и Невис", "Сент-Люсия", "Сербия", "Сингапур", "Сирия", "Словакия", "Словения",
    "Соломоновы Острова", "Сомали", "Судан", "Суринам", "Сьерра-Леоне", "Таджикистан", "Таиланд",
    "Танзания", "Того", "Тонга", "Тринидад и Тобаго", "Тувалу", "Тунис", "Туркменистан", "Турция",
    "Уганда", "Узбекистан", "Украина", "Уругвай", "Фиджи", "Филиппины", "Финляндия", "Франция",
    "Хорватия", "Центральноафриканская Республика", "Чад", "Черногория", "Чехия", "Чили",
    "Швейцария", "Швеция", "Шри-Ланка", "Эквадор", "Экваториальная Гвинея", "Эритрея", "Эсватини",
    "Эстония", "Эфиопия", "Южно-Африканская Республика", "Южный Судан", "Ямайка", "Япония"
]
